# Train-test Split and Cleaning
Splitting the combined file. Cleaning of combined dataset.

## 1. Imports & Setup

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.spatial import cKDTree

## 2. Preliminary Cleaning Before Split

In [16]:
df = pd.read_csv("combined.csv")

# Drop columns that are leakage or not useful for prediction
df = df.drop(columns=["date", "origin_lat", "origin_lon", "dest_lat", "dest_lon"])

# Fill NaN values in weather columns with median
weather_feature_cols = [c for c in df.columns if (c.startswith("origin_") or c.startswith("dest_")) and df[c].dtype != object]
df[weather_feature_cols] = df[weather_feature_cols].fillna(df[weather_feature_cols].median())    

# Encode carrier code
le = LabelEncoder()
df["Carrier Code"] = le.fit_transform(df["Carrier Code"])
carrier_mapping = dict(zip(le.transform(le.classes_), le.classes_))
    
# Encode origin and destination with shared mapping
all_airports = pd.Series(pd.concat([df["origin_code"], df["destination_code"]]).unique())
le.fit(all_airports)
airport_mapping = dict(zip(le.transform(le.classes_), le.classes_))
df["origin_code"] = le.transform(df["origin_code"])
df["destination_code"] = le.transform(df["destination_code"])
    
print("Carrier mapping:", carrier_mapping)
print("Airport mapping:", airport_mapping)

Carrier mapping: {0: 'AA', 1: 'DL', 2: 'UA', 3: 'WN'}
Airport mapping: {0: 'ABQ', 1: 'AGS', 2: 'ALB', 3: 'AMA', 4: 'ANC', 5: 'ATL', 6: 'ATW', 7: 'AUS', 8: 'AVL', 9: 'AVP', 10: 'BDL', 11: 'BFL', 12: 'BGR', 13: 'BHM', 14: 'BIL', 15: 'BIS', 16: 'BLI', 17: 'BNA', 18: 'BOI', 19: 'BOS', 20: 'BQN', 21: 'BTR', 22: 'BTV', 23: 'BUF', 24: 'BUR', 25: 'BWI', 26: 'BZN', 27: 'CAE', 28: 'CHA', 29: 'CHO', 30: 'CHS', 31: 'CID', 32: 'CLE', 33: 'CLT', 34: 'CMH', 35: 'COS', 36: 'CRP', 37: 'CVG', 38: 'DAB', 39: 'DAL', 40: 'DAY', 41: 'DCA', 42: 'DEN', 43: 'DFW', 44: 'DLH', 45: 'DRO', 46: 'DSM', 47: 'DTW', 48: 'ECP', 49: 'EGE', 50: 'ELP', 51: 'EUG', 52: 'EVV', 53: 'EWR', 54: 'EYW', 55: 'FAI', 56: 'FAR', 57: 'FAT', 58: 'FAY', 59: 'FCA', 60: 'FLL', 61: 'FSD', 62: 'GEG', 63: 'GJT', 64: 'GNV', 65: 'GPT', 66: 'GRB', 67: 'GRR', 68: 'GSO', 69: 'GSP', 70: 'GTF', 71: 'GUC', 72: 'GUM', 73: 'HDN', 74: 'HNL', 75: 'HOU', 76: 'HPN', 77: 'HRL', 78: 'HSV', 79: 'IAD', 80: 'IAH', 81: 'ICT', 82: 'IDA', 83: 'ILM', 84: 'IND', 85:

In [17]:
cols = ['Carrier Code', 'destination_code', 'origin_code', 'scheduled_hour']

In [ ]:
def one_hot(df, cols):
    """
    df: pandas DataFrame
    cols: a list of columns to encode 
    return a DataFrame with one-hot encoding
    """
    for each in cols:
        dummies = pd.get_dummies(df[each], prefix=each, drop_first=False).astype(int) # creates the one-hot encoding cols
        df = pd.concat([df, dummies], axis=1)

    return df

cleaned_df = one_hot(df, cols)

## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## 3. Cleaning combined data

In [ ]:
def clean(df):
    # Drop any remaining NaN rows
    cleaned_df = cleaned_df.dropna()

X_train = clean(X_train)
X_test = clean(X_test)
y_train = clean(y_train)
y_test = clean(y_test)